In [4]:
from pyspark.sql import functions as F
df_dates = spark.read.format("delta").load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Gold.Lakehouse/Tables/dbo/dim_date")


StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84a27cb3-6dc4-43ac-a23b-e5132affd36d)

In [5]:
df_date = df_dates.withColumn(
    "fiscal_year",
    F.when(
        F.month("date") >= 4,
        F.year("date") + 1
    ).otherwise(
        F.year("date")
    )
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 7, Finished, Available, Finished, False)

In [7]:
df_date = df_date.withColumn(
    "fiscal_year_label",
    F.concat(
        F.lit("FY"),
        F.col("fiscal_year").cast("string")
    )
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 9, Finished, Available, Finished, False)

In [8]:
df_date = df_date.withColumn(
    "fiscal_quarter",
    F.when(F.month("date").between(4, 6), "Q1")
     .when(F.month("date").between(7, 9), "Q2")
     .when(F.month("date").between(10, 12), "Q3")
     .otherwise("Q4")
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 10, Finished, Available, Finished, False)

In [9]:
df_date = df_date.withColumn(
    "fiscal_quarter_number",
    F.when(F.month("date").between(4, 6), 1)
     .when(F.month("date").between(7, 9), 2)
     .when(F.month("date").between(10, 12), 3)
     .otherwise(4)
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 11, Finished, Available, Finished, False)

In [10]:
df_date = df_date.withColumn(
    "fiscal_month_number",
    F.when(
        F.month("date") >= 4,
        F.month("date") - 3
    ).otherwise(
        F.month("date") + 9
    )
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 12, Finished, Available, Finished, False)

In [11]:
df_date = df_date.withColumn(
    "fiscal_month",
    F.date_format("date", "MMM")
)

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 13, Finished, Available, Finished, False)

In [12]:
df_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 14, Finished, Available, Finished, False)

In [13]:
%%sql
SELECT
    date,
    fiscal_year,
    fiscal_year_label,
    fiscal_quarter,
    fiscal_month_number,
    fiscal_month
FROM dim_date
ORDER BY date
LIMIT 20;

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 15, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 6 fields>

In [14]:
%%sql
SELECT
    date,
    fiscal_year_label,
    fiscal_quarter,
    fiscal_month_number
FROM dim_date
WHERE date IN (
    '2026-03-31',
    '2026-04-01',
    '2026-06-30',
    '2026-07-01',
    '2026-09-30',
    '2026-10-01'
)
ORDER BY date;

StatementMeta(, 029265bd-b9a4-416f-8513-5d684145e128, 16, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>